# Lesson 2 — Noise Models and Quantum Channels

**Quantum Computing with Qiskit II**

This notebook accompanies Lesson 2. Work through the cells in order.

**Topics covered:**
- Kraus operators and the completeness relation
- Bit-flip, depolarizing, and amplitude-damping channels
- Building custom noise models in AerSimulator
- Noise profiling from real device calibration data (FakeSherbrooke)

In [ ]:
!pip install qiskit qiskit-aer qiskit-ibm-runtime --quiet

In [ ]:
import qiskit
import qiskit_aer
import qiskit_ibm_runtime

print("Qiskit:             ", qiskit.__version__)
print("Qiskit Aer:         ", qiskit_aer.__version__)
print("Qiskit IBM Runtime: ", qiskit_ibm_runtime.__version__)

In [ ]:
# Core imports used throughout the notebook
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import DensityMatrix, Statevector
from qiskit_aer import AerSimulator
from qiskit_aer.noise import (
    NoiseModel,
    depolarizing_error,
    amplitude_damping_error,
    thermal_relaxation_error,
    ReadoutError,
)
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

print("Imports complete.")

## 1. Quantum Channels and Kraus Operators

A **quantum channel** is a linear map $\mathcal{E}$ on density matrices that is completely positive
and trace-preserving (CPTP). Every CPTP map can be written as:

$$\mathcal{E}(\rho) = \sum_k K_k \rho K_k^\dagger$$

where the **Kraus operators** $\{K_k\}$ satisfy $\sum_k K_k^\dagger K_k = I$.

Unitary evolution $\rho \to U\rho U^\dagger$ is the special case with a single Kraus operator $K_0 = U$.

In [ ]:
def apply_channel(rho_dm, kraus_ops):
    """Apply E(rho) = sum_k K_k @ rho @ K_k†."""
    out = np.zeros_like(rho_dm.data, dtype=complex)
    for K in kraus_ops:
        out += K @ rho_dm.data @ K.conj().T
    return DensityMatrix(out)


def check_completeness(kraus_ops):
    """Verify sum_k K†K = I (required for trace preservation)."""
    n = kraus_ops[0].shape[0]
    total = sum(K.conj().T @ K for K in kraus_ops)
    return np.allclose(total, np.eye(n))


# Verify completeness for a sample bit-flip channel (p=0.2)
p = 0.2
K0 = np.sqrt(1 - p) * np.eye(2)
K1 = np.sqrt(p) * np.array([[0, 1], [1, 0]])

print("K0†K0 + K1†K1:")
print(np.round(K0.conj().T @ K0 + K1.conj().T @ K1, 6))
print(f"Completeness check: {check_completeness([K0, K1])}")

## 2. The Bit-Flip Channel

The bit-flip channel applies $X$ with probability $p$ and identity with probability $1-p$:

$$\mathcal{E}(\rho) = (1-p)\rho + p\,X\rho X$$

Bloch vector transformation: $r_x \to r_x$, $r_y \to (1-2p)r_y$, $r_z \to (1-2p)r_z$.

Key property: $|{+}\rangle$ is an eigenstate of $X$, so **bit-flip has no effect on $|{+}\rangle$**.
This illustrates why encoding quantum information in the correct basis can protect against specific noise types.

In [ ]:
def bit_flip_kraus(p):
    K0 = np.sqrt(1 - p) * np.eye(2)
    K1 = np.sqrt(p) * np.array([[0, 1], [1, 0]])   # X
    return [K0, K1]


p = 0.3
rho_0    = DensityMatrix(Statevector.from_label('0'))
rho_plus = DensityMatrix(Statevector.from_label('+'))

out_0    = apply_channel(rho_0,    bit_flip_kraus(p))
out_plus = apply_channel(rho_plus, bit_flip_kraus(p))

print(f"Bit-flip (p={p}) on |0>:")
print(np.round(out_0.data, 4))
print(f"Purity: {out_0.purity().real:.4f}")
print()
print(f"Bit-flip (p={p}) on |+>:")
print(np.round(out_plus.data, 4))
print(f"Purity: {out_plus.purity().real:.4f}  (immune: X|+> = |+>)")

In [ ]:
# Plot purity of E(|0>) under bit-flip vs p
# Analytical purity: (1-p)^2 + p^2
p_vals = np.linspace(0, 1, 200)
purity_bf_0    = [apply_channel(rho_0,    bit_flip_kraus(p)).purity().real for p in p_vals]
purity_bf_plus = [apply_channel(rho_plus, bit_flip_kraus(p)).purity().real for p in p_vals]
analytic_bf    = (1 - p_vals)**2 + p_vals**2

plt.figure(figsize=(7, 3))
plt.plot(p_vals, purity_bf_0,    label='Input |0⟩',     color='steelblue', linewidth=2)
plt.plot(p_vals, purity_bf_plus, label='Input |+⟩ (immune)', color='tomato', linewidth=2, linestyle='--')
plt.plot(p_vals, analytic_bf,    label='Analytic (1-p)²+p²', color='black', linewidth=1, linestyle=':')
plt.axhline(0.5, color='gray', linestyle=':', linewidth=0.8)
plt.xlabel("Bit-flip probability p")
plt.ylabel("Output purity")
plt.title("Bit-flip channel: purity vs p")
plt.legend()
plt.tight_layout()
plt.show()

## 3. The Depolarizing Channel

The depolarizing channel applies $X$, $Y$, or $Z$ each with probability $p/3$:

$$\mathcal{E}(\rho) = (1-p)\rho + \frac{p}{3}(X\rho X + Y\rho Y + Z\rho Z) = \left(1-\tfrac{4p}{3}\right)\rho + \tfrac{4p}{3}\cdot\frac{I}{2}$$

All Bloch vector components shrink by $(1-4p/3)$. At $p = 3/4$ the state becomes $I/2$
regardless of input. No basis is immune.

In [ ]:
def depolarizing_kraus(p):
    X = np.array([[0, 1], [1, 0]])
    Y = np.array([[0, -1j], [1j, 0]])
    Z = np.array([[1, 0], [0, -1]])
    K0 = np.sqrt(1 - p)     * np.eye(2)
    K1 = np.sqrt(p / 3)     * X
    K2 = np.sqrt(p / 3)     * Y
    K3 = np.sqrt(p / 3)     * Z
    return [K0, K1, K2, K3]


p = 0.3
out_dep = apply_channel(rho_0, depolarizing_kraus(p))

print(f"Depolarizing (p={p}) on |0>:")
print(np.round(out_dep.data, 4))
print(f"Purity: {out_dep.purity().real:.4f}")
print()

# At p=0.75 the output should be I/2 for any input
out_dep_max = apply_channel(rho_0, depolarizing_kraus(0.75))
print("Depolarizing (p=0.75) on |0>:")
print(np.round(out_dep_max.data, 4))
print(f"Purity: {out_dep_max.purity().real:.4f}  (maximally mixed)")

In [ ]:
# Verify Bloch vector shrinkage: depolarizing on a pure state
# Analytical formula: Bloch vector scales by (1 - 4p/3)

rho_y = DensityMatrix(np.array([[0.5, -0.5j], [0.5j, 0.5]]))   # |+i> = (|0>+i|1>)/sqrt(2)

print("Input state |+i> (Bloch vector points along +y):")
print(np.round(rho_y.data, 4))
print()

for p_test in [0.0, 0.1, 0.3, 0.75]:
    out = apply_channel(rho_y, depolarizing_kraus(p_test))
    scale = 1 - 4 * p_test / 3
    off_diag = out.data[0, 1]
    print(f"p={p_test:.2f}: off-diagonal = {off_diag:.4f}  "
          f"(expected {-0.5j * scale:.4f})  scale factor = {scale:.4f}")

## 4. The Amplitude-Damping Channel

Amplitude damping models **T1 relaxation**: the excited state $|1\rangle$ spontaneously decays
to the ground state $|0\rangle$ with probability $\gamma$.

$$K_0 = \begin{pmatrix}1 & 0 \\ 0 & \sqrt{1-\gamma}\end{pmatrix} \qquad
K_1 = \begin{pmatrix}0 & \sqrt{\gamma} \\ 0 & 0\end{pmatrix}$$

For a gate of duration $\Delta t$ on a device with relaxation time $T_1$:
$\gamma = 1 - e^{-\Delta t / T_1}$

The ground state $|0\rangle$ is a dark state (unchanged). Coherences decay as $\sqrt{1-\gamma}$ per step,
which is faster than the population decay $(1-\gamma)$.

In [ ]:
def amplitude_damping_kraus(gamma):
    K0 = np.array([[1, 0],
                   [0, np.sqrt(1 - gamma)]], dtype=complex)
    K1 = np.array([[0, np.sqrt(gamma)],
                   [0, 0]], dtype=complex)
    return [K0, K1]


gamma = 0.3
rho_1 = DensityMatrix(Statevector.from_label('1'))

out_ad_1 = apply_channel(rho_1, amplitude_damping_kraus(gamma))
out_ad_0 = apply_channel(rho_0, amplitude_damping_kraus(gamma))

print(f"Amplitude-damping (gamma={gamma}) on |1>:")
print(np.round(out_ad_1.data, 4))
print(f"Purity: {out_ad_1.purity().real:.4f}")
print()
print(f"Amplitude-damping (gamma={gamma}) on |0>:")
print(np.round(out_ad_0.data, 4))
print(f"Purity: {out_ad_0.purity().real:.4f}  (dark state: unchanged)")

In [ ]:
# Apply amplitude damping repeatedly to simulate T1 decay over time
# gamma_step corresponds to a single gate duration (e.g. 50 ns)

T1 = 150e-6          # 150 µs
gate_time = 50e-9    # 50 ns per step
gamma_step = 1 - np.exp(-gate_time / T1)

n_steps = 500
rho_current = DensityMatrix(Statevector.from_label('1'))
pop_1 = [rho_current.data[1, 1].real]   # P(|1>)

for _ in range(n_steps):
    rho_current = apply_channel(rho_current, amplitude_damping_kraus(gamma_step))
    pop_1.append(rho_current.data[1, 1].real)

times_us = np.arange(n_steps + 1) * gate_time * 1e6
analytic = np.exp(-times_us * 1e-6 / T1)   # e^{-t/T1}

plt.figure(figsize=(7, 3))
plt.plot(times_us, pop_1,    label='Repeated K0,K1 application', color='steelblue', linewidth=2)
plt.plot(times_us, analytic, label=f'Analytic e^{{-t/T1}} (T1={T1*1e6:.0f} µs)',
         color='black', linewidth=1.5, linestyle='--')
plt.xlabel("Time (µs)")
plt.ylabel("P(|1⟩)")
plt.title("T1 decay: amplitude-damping channel applied repeatedly")
plt.legend()
plt.tight_layout()
plt.show()

print(f"gamma per step: {gamma_step:.2e}")
print(f"Repeated application matches analytic T1 decay.")

## 5. Comparing All Three Channels

Plotting purity against the error parameter for the same input state reveals each channel's signature.

In [ ]:
# Input: |1> — all three channels have a clear effect
rho_input = DensityMatrix(Statevector.from_label('1'))

params = np.linspace(0, 0.75, 120)

purity_bf  = [apply_channel(rho_input, bit_flip_kraus(p)).purity().real           for p in params]
purity_dep = [apply_channel(rho_input, depolarizing_kraus(p)).purity().real       for p in params]
purity_ad  = [apply_channel(rho_input, amplitude_damping_kraus(g)).purity().real  for g in params]

plt.figure(figsize=(8, 3.5))
plt.plot(params, purity_bf,  label='Bit-flip',         color='steelblue', linewidth=2)
plt.plot(params, purity_dep, label='Depolarizing',      color='tomato',    linewidth=2)
plt.plot(params, purity_ad,  label='Amplitude-damping', color='seagreen',  linewidth=2)
plt.axhline(0.5, color='gray', linestyle=':', linewidth=0.8, label='Purity = 0.5')
plt.xlabel("Error parameter (p or γ)")
plt.ylabel("Output purity")
plt.title("Purity of $\\mathcal{E}(|1\\rangle\\langle 1|)$ under three channels")
plt.legend()
plt.tight_layout()
plt.show()

print("Observations:")
print("  Bit-flip and amplitude-damping give the same purity on |1> (both produce diagonal states).")
print("  Depolarizing is more damaging at the same p because it applies three independent errors.")
print("  Amplitude-damping at gamma=1 returns to the pure state |0>: purity=1 again.")

## 6. Building Noise Models in AerSimulator

Qiskit Aer's `NoiseModel` attaches error channels to specific gate names and qubit positions.
It accepts the same Kraus-based error objects used above, packaged as `QuantumError` instances
via the constructor functions (`depolarizing_error`, `amplitude_damping_error`, etc.).

In [ ]:
# Build a custom noise model with realistic error rates
p_1q   = 0.001    # 0.1%  single-qubit gate error
p_2q   = 0.010    # 1.0%  two-qubit gate error
p_ro_0 = 0.010    # P(read 1 | prepared 0)
p_ro_1 = 0.030    # P(read 0 | prepared 1)  -- higher: excited state decays during readout

noise_model = NoiseModel()

# Single-qubit depolarizing on sx, x, rz
dep_1q = depolarizing_error(p_1q, 1)
noise_model.add_all_qubit_quantum_error(dep_1q, ['sx', 'x', 'rz'])

# Two-qubit depolarizing on ecr / cx
dep_2q = depolarizing_error(p_2q, 2)
noise_model.add_all_qubit_quantum_error(dep_2q, ['ecr', 'cx'])

# Asymmetric readout error: ReadoutError([[P(0|0), P(0|1)], [P(1|0), P(1|1)]])
ro_err = ReadoutError([[1 - p_ro_0,     p_ro_1],
                       [    p_ro_0, 1 - p_ro_1]])
noise_model.add_all_qubit_readout_error(ro_err)

print("Basis gates with noise:", noise_model.basis_gates)
print("Gates with errors:     ", noise_model.noise_instructions)

In [ ]:
# Run a Bell state circuit: ideal vs custom noise model
qc_bell = QuantumCircuit(2, 2)
qc_bell.h(0)
qc_bell.cx(0, 1)
qc_bell.measure([0, 1], [0, 1])

shots = 8192

counts_ideal = AerSimulator().run(qc_bell, shots=shots).result().get_counts()
counts_noisy = AerSimulator(noise_model=noise_model).run(qc_bell, shots=shots).result().get_counts()

print(f"{'State':>6}  {'Ideal':>8}  {'Noisy':>8}")
print("-" * 26)
for state in ['00', '01', '10', '11']:
    p_ideal = counts_ideal.get(state, 0) / shots
    p_noisy = counts_noisy.get(state, 0) / shots
    print(f"  |{state}>  {p_ideal:>8.4f}  {p_noisy:>8.4f}")

print()
print("Noise transfers probability from |00> and |11> into error states |01> and |10>.")

In [ ]:
# Thermal relaxation error: more accurate than flat depolarizing
# Combines T1 (amplitude damping) and T2 (dephasing) in the correct proportion.

T1        = 150e-6    # 150 µs
T2        = 100e-6    # T2 < 2*T1 (includes pure dephasing)
gate_time = 50e-9     # 50 ns for a single-qubit gate

therm_err = thermal_relaxation_error(T1, T2, gate_time)

# Build a noise model using thermal relaxation instead of depolarizing
noise_model_thermal = NoiseModel()
noise_model_thermal.add_all_qubit_quantum_error(therm_err, ['sx', 'x', 'rz'])
noise_model_thermal.add_all_qubit_quantum_error(dep_2q, ['cx', 'ecr'])
noise_model_thermal.add_all_qubit_readout_error(ro_err)

counts_thermal = AerSimulator(noise_model=noise_model_thermal).run(qc_bell, shots=shots).result().get_counts()

print(f"Bell state P('00') with thermal noise: {counts_thermal.get('00', 0) / shots:.4f}")
print(f"Bell state P('00') with depol  noise:  {counts_noisy.get('00', 0) / shots:.4f}")
print("(Both should be close to 0.5 for well-chosen parameters.")

## 7. Noise Profiling from Real Device Data

`NoiseModel.from_backend(backend)` reads a device's calibration snapshot and constructs a
complete noise model automatically. `FakeSherbrooke` ships with calibrated T1, T2, gate
error rates, and readout errors for all 127 qubits.

In [ ]:
backend = FakeSherbrooke()
noise_model_device = NoiseModel.from_backend(backend)

print(f"Backend: {backend.name}")
print(f"Qubits:  {backend.num_qubits}")
print(f"Noisy gates: {noise_model_device.noise_instructions}")

In [ ]:
# Inspect T1 and T2 times for the first 8 qubits
target = backend.target

print(f"{'Qubit':>6}  {'T1 (µs)':>10}  {'T2 (µs)':>10}  {'T2/T1':>8}")
print("-" * 40)
for i in range(8):
    qp = target.qubit_properties[i]
    t1 = qp.t1 * 1e6 if qp.t1 is not None else float('nan')
    t2 = qp.t2 * 1e6 if qp.t2 is not None else float('nan')
    ratio = t2 / t1 if (qp.t1 and qp.t2) else float('nan')
    print(f"{i:>6}  {t1:>10.1f}  {t2:>10.1f}  {ratio:>8.3f}")

print()
print("T2 <= 2*T1 always (pure dephasing drives T2 below the T1 limit).")

In [ ]:
# Inspect single-qubit (sx) and two-qubit (ecr/cx) gate error rates

# sx errors for first 8 qubits
print("sx gate error rates (first 8 qubits):")
for i in range(8):
    props = target['sx'][(i,)]
    if props is not None and props.error is not None:
        print(f"  qubit {i}: {props.error:.5f}  ({props.error * 100:.3f}%)")
print()

# Two-qubit gate errors: best and worst pairs
twoq_name = 'ecr' if 'ecr' in target.operation_names else 'cx'
twoq_errors = [
    (qargs, props.error)
    for qargs, props in target[twoq_name].items()
    if props is not None and props.error is not None
]
twoq_errors.sort(key=lambda x: x[1])

print(f"5 best {twoq_name.upper()} pairs:")
for qargs, err in twoq_errors[:5]:
    print(f"  {twoq_name}({qargs[0]:>3},{qargs[1]:>3}): {err:.5f}  ({err * 100:.3f}%)")
print()
print(f"5 worst {twoq_name.upper()} pairs:")
for qargs, err in twoq_errors[-5:]:
    print(f"  {twoq_name}({qargs[0]:>3},{qargs[1]:>3}): {err:.5f}  ({err * 100:.3f}%)")

In [ ]:
# Plot the distribution of ECR/CX error rates across the device
all_errors = [err for _, err in twoq_errors]

plt.figure(figsize=(7, 3))
plt.hist(all_errors, bins=30, color='steelblue', edgecolor='white', linewidth=0.5)
plt.xlabel(f"{twoq_name.upper()} error rate")
plt.ylabel("Number of qubit pairs")
plt.title(f"{twoq_name.upper()} gate error distribution — FakeSherbrooke (127 qubits)")
plt.axvline(np.median(all_errors), color='tomato', linewidth=1.5,
            label=f"Median: {np.median(all_errors):.4f}")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Mean  {twoq_name.upper()} error: {np.mean(all_errors):.5f}")
print(f"Median {twoq_name.upper()} error: {np.median(all_errors):.5f}")
print(f"Max    {twoq_name.upper()} error: {np.max(all_errors):.5f}")

In [ ]:
# Compare ideal, custom depolarizing, and device-calibrated noise models
# on a 2-qubit Grover circuit (target='11', k=1)

def grover_2qubit(target='11'):
    """Grover circuit for n=2, one iteration."""
    qc = QuantumCircuit(2, 2)
    qc.h([0, 1])
    # Phase oracle: flip sign of |target>
    if target[1] == '0': qc.x(0)
    if target[0] == '0': qc.x(1)
    qc.cz(0, 1)
    if target[1] == '0': qc.x(0)
    if target[0] == '0': qc.x(1)
    # Diffuser: 2|s><s| - I
    qc.h([0, 1])
    qc.x([0, 1])
    qc.cz(0, 1)
    qc.x([0, 1])
    qc.h([0, 1])
    qc.measure([0, 1], [0, 1])
    return qc


target = '11'
qc_grover = grover_2qubit(target)
shots = 8192

# Ideal
counts_ideal  = AerSimulator().run(qc_grover, shots=shots).result().get_counts()

# Custom depolarizing
counts_custom = AerSimulator(noise_model=noise_model).run(qc_grover, shots=shots).result().get_counts()

# Device-calibrated (transpile to backend native gates first)
t_grover = transpile(qc_grover, backend=backend, optimization_level=3, seed_transpiler=42)
counts_device = AerSimulator(noise_model=noise_model_device).run(
    t_grover, shots=shots).result().get_counts()

print(f"P('{target}') — Ideal:          {counts_ideal.get(target, 0) / shots:.3f}")
print(f"P('{target}') — Custom (depol): {counts_custom.get(target, 0) / shots:.3f}")
print(f"P('{target}') — Device (FakeSherbrooke): {counts_device.get(target, 0) / shots:.3f}")
print()
print("Device noise is more pessimistic: it captures T1/T2 decoherence and")
print("crosstalk that a flat depolarizing rate cannot represent.")

## 8. Exercises

Work through each exercise in the cell below it. Expected results are given so you can check.

### Exercise 1: Phase-flip channel

The **phase-flip channel** applies a $Z$ error with probability $p$:
$$\mathcal{E}(\rho) = (1-p)\rho + p\,Z\rho Z$$

1. Implement `phase_flip_kraus(p)` following the pattern from bit-flip.
2. Show that $|0\rangle$ is immune to the phase-flip channel.
3. Apply the channel to $|{+}\rangle$ for $p = 0.3$ and report the output density matrix and purity.

*Expected: $|{+}\rangle$ output has off-diagonal entries $(1-2p)\times 0.5 = 0.2$, purity $\approx 0.58$*

In [ ]:
# Exercise 1 — Phase-flip channel

def phase_flip_kraus(p):
    # TODO: implement
    pass


# p = 0.3
# rho_plus = DensityMatrix(Statevector.from_label('+'))
# out_pf_plus = apply_channel(rho_plus, phase_flip_kraus(p))
# print(np.round(out_pf_plus.data, 4))
# print(f"Purity: {out_pf_plus.purity().real:.4f}")

### Exercise 2: T1 decay rate from backend data

1. Use the FakeSherbrooke `target` object to retrieve T1 for all 127 qubits.
2. Compute the amplitude-damping parameter $\gamma$ for a 50 ns gate on each qubit
   using $\gamma = 1 - e^{-\Delta t / T_1}$.
3. Find the qubit with the largest $\gamma$ (worst T1) and the smallest $\gamma$ (best T1).
4. What is the ratio between worst and best $\gamma$?

*Expected: ratio is approximately 3 to 6 depending on the calibration snapshot.*

In [ ]:
# Exercise 2 — T1 decay rates

gate_time_ns = 50e-9   # 50 ns

# TODO: iterate over target.qubit_properties, compute gamma for each qubit
# gammas = []
# for i, qp in enumerate(target.qubit_properties):
#     if qp.t1 is not None:
#         gamma = 1 - np.exp(-gate_time_ns / qp.t1)
#         gammas.append((i, gamma))

# TODO: find best and worst, compute ratio
# gammas.sort(key=lambda x: x[1])
# best  = gammas[0]
# worst = gammas[-1]
# print(f"Best  qubit: {best[0]},  gamma = {best[1]:.4e}")
# print(f"Worst qubit: {worst[0]}, gamma = {worst[1]:.4e}")
# print(f"Ratio worst/best: {worst[1] / best[1]:.1f}")

### Exercise 3: Custom noise model with thermal relaxation

Replace the flat `depolarizing_error` on single-qubit gates with per-qubit
`thermal_relaxation_error`, using the T1 and T2 values read from FakeSherbrooke.

1. Build a 5-qubit `NoiseModel` where each qubit has its own thermal relaxation error
   on `sx` gates, using `gate_time = 50e-9` and the FakeSherbrooke T1/T2 values.
   Use `noise_model.add_quantum_error(err, 'sx', [qubit_index])` for per-qubit errors.
2. Run the Bell state circuit on qubits 0 and 1 under this model.
3. Compare `P('00')` with the flat depolarizing result from cell-19.

*The thermal model should give a result closer to the device-calibrated `noise_model_device`.*

In [ ]:
# Exercise 3 — Per-qubit thermal relaxation noise model

gate_time = 50e-9
n_qubits  = 5

# TODO: build a noise model with per-qubit thermal errors on sx
# noise_model_ex3 = NoiseModel()
# for i in range(n_qubits):
#     qp = target.qubit_properties[i]
#     if qp.t1 is not None and qp.t2 is not None:
#         err = thermal_relaxation_error(qp.t1, qp.t2, gate_time)
#         noise_model_ex3.add_quantum_error(err, 'sx', [i])

# TODO: run Bell circuit and compare P('00') with flat depolarizing result

## Summary

In this lesson you:

- Derived the Kraus representation $\mathcal{E}(\rho) = \sum_k K_k \rho K_k^\dagger$ and verified the
  completeness relation $\sum_k K_k^\dagger K_k = I$ by direct computation.
- Implemented three fundamental channels: **bit-flip** (preserves the $x$-basis), **depolarizing**
  (shrinks the entire Bloch vector uniformly), and **amplitude-damping** (models T1 decay,
  with $|0\rangle$ as a dark state). Confirmed that repeated amplitude-damping steps reproduce
  the analytic $e^{-t/T_1}$ decay curve.
- Built a custom `NoiseModel` in AerSimulator using `depolarizing_error`, `thermal_relaxation_error`,
  and `ReadoutError`, attaching errors to specific gate names and all qubits.
- Used `NoiseModel.from_backend(FakeSherbrooke())` to build a device-calibrated noise model,
  inspected T1/T2 times and gate error rates from the `target` object, and confirmed that
  two-qubit gate errors are roughly 10 to 100 times larger than single-qubit gate errors.
- Ran a Grover benchmark under all three conditions — ideal, flat depolarizing, and device-calibrated
  — and observed that the device model is consistently more pessimistic due to its richer noise structure.

**Lesson 3** introduces error mitigation: zero-noise extrapolation runs the same circuit at
amplified noise levels and extrapolates back to zero, and probabilistic error cancellation
inverts the noise channel in post-processing. Both are available through the Qiskit Runtime
`Estimator` primitive.